# ⚙️ Module 5 — Training Best Practices
### Introduction to Deep Learning | AgenticLabs.ng

---

## 🎯 Learning Objectives
- Diagnose **overfitting** and **underfitting** from loss curves
- Apply key regularisation techniques: **Dropout**, **Weight Decay (L2)**
- Understand and use **Batch Normalisation**
- Use **learning rate schedulers** to improve convergence
- Implement **early stopping** to avoid overfitting
- Use **TensorBoard** to track experiments in real time
- Learn the full production training checklist

---

## 5.1 — Diagnosing Your Model: Overfitting vs Underfitting

The training/validation loss curve is your most important diagnostic tool.

```
Underfitting:  High train loss + High val loss  → Model too simple, train longer
Good fit:      Low train loss + Low val loss    → 
Overfitting:   Low train loss + High val loss   → Regularise!
```


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Simulate the three scenarios ──────────────────────────
epochs = np.arange(1, 51)

def noisy(base, noise=0.02):
    return base + np.random.randn(len(epochs)) * noise

scenarios = {
    "Underfitting": {
        "train": 0.8 - 0.3*(1-np.exp(-epochs/30)),
        "val":   0.85 - 0.28*(1-np.exp(-epochs/30)),
        "color": "blue"
    },
    "Good fit": {
        "train": 0.8 - 0.6*(1-np.exp(-epochs/15)),
        "val":   0.82 - 0.55*(1-np.exp(-epochs/15)),
        "color": "green"
    },
    "Overfitting": {
        "train": 0.8 - 0.75*(1-np.exp(-epochs/10)),
        "val":   np.clip(0.78 - 0.3*(1-np.exp(-epochs/10)) + 0.006*epochs, 0.2, 0.9),
        "color": "red"
    }
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, sc) in enumerate(scenarios.items()):
    axes[i].plot(epochs, noisy(sc["train"]), color=sc["color"], lw=2, label='Train loss')
    axes[i].plot(epochs, noisy(sc["val"]),   color=sc["color"], lw=2, linestyle='--', label='Val loss')
    axes[i].set_title(name, fontsize=13, fontweight='bold', color=sc["color"])
    axes[i].set_xlabel("Epoch"); axes[i].set_ylabel("Loss")
    axes[i].legend(); axes[i].grid(alpha=0.3); axes[i].set_ylim(0, 1)

plt.suptitle("Learning Curve Diagnostic Patterns", fontsize=14)
plt.tight_layout(); plt.show()


## 5.2 — Regularisation Techniques

In [ ]:
# ── Compare: No Dropout vs Dropout vs L2 ──────────────────
# We use a deliberately overfit setup (small dataset, large model)
transform = transforms.Compose([transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])

full_train = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
# Deliberately use a tiny subset to induce overfitting
small_train, _   = random_split(full_train, [2000, 48000])
test_set         = datasets.CIFAR10('./data', train=False, transform=transform)

small_loader = DataLoader(small_train, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set,   batch_size=64, shuffle=False)

def make_model(dropout_rate=0.0, weight_decay=0.0, name=""):
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(3*32*32, 1024), nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(1024, 512),     nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(512, 10)
    ).to(device)
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    return model, opt

def run_experiment(model, opt, name, epochs=20):
    criterion = nn.CrossEntropyLoss()
    tr_losses, te_losses, tr_accs, te_accs = [], [], [], []
    for epoch in range(epochs):
        model.train()
        tl, tc, tt = 0, 0, 0
        for X, y in small_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward(); opt.step()
            tl += loss.item()
            tc += (out.argmax(1)==y).sum().item(); tt += y.size(0)
        model.eval()
        el, ec, et = 0, 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                el += criterion(out, y).item()
                ec += (out.argmax(1)==y).sum().item(); et += y.size(0)
        tr_losses.append(tl/len(small_loader)); te_losses.append(el/len(test_loader))
        tr_accs.append(tc/tt*100);             te_accs.append(ec/et*100)
    return tr_losses, te_losses, tr_accs, te_accs

m1, o1 = make_model(dropout_rate=0.0, weight_decay=0.0, name="No regularisation")
m2, o2 = make_model(dropout_rate=0.5, weight_decay=0.0, name="Dropout 0.5")
m3, o3 = make_model(dropout_rate=0.0, weight_decay=1e-3, name="L2 weight decay")

print("Training 3 models (no reg | dropout | L2)...")
r1 = run_experiment(m1, o1, "No reg",  epochs=20)
r2 = run_experiment(m2, o2, "Dropout", epochs=20)
r3 = run_experiment(m3, o3, "L2",      epochs=20)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, 21)
for (tr, te, _, __), label, c in [(r1,"No reg","red"),(r2,"Dropout","blue"),(r3,"L2","green")]:
    axes[0].plot(ep, tr, '-', color=c, alpha=0.7, label=f'{label} Train')
    axes[0].plot(ep, te, '--', color=c, lw=2, label=f'{label} Val')

for (*__, tr_a, te_a), label, c in [(r1,"No reg","red"),(r2,"Dropout","blue"),(r3,"L2","green")]:
    axes[1].plot(ep, te_a, '-o', color=c, ms=4, label=label)

axes[0].set_title("Loss: No Reg vs Dropout vs L2"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_title("Test Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle("Effect of Regularisation on Overfitting", fontsize=13)
plt.tight_layout(); plt.show()


## 5.3 — Batch Normalisation: Why It Works

Batch norm normalises layer inputs to have zero mean and unit variance within each mini-batch. This reduces the effect of weight initialisation, accelerates training, and acts as mild regularisation.

In [ ]:
# ── BatchNorm impact on training speed ────────────────────
class NetNoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 512), nn.ReLU(),
            nn.Linear(512, 256),  nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)

class NetWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256),  nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)

def quick_train(model, epochs=10):
    opt = optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()
    accs = []
    for _ in range(epochs):
        model.train()
        for X, y in small_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad(); loss=crit(model(X),y); loss.backward(); opt.step()
        model.eval(); c=t=0
        with torch.no_grad():
            for X,y in test_loader:
                X,y=X.to(device),y.to(device)
                c+=(model(X).argmax(1)==y).sum().item(); t+=y.size(0)
        accs.append(c/t*100)
    return accs

nobn = NetNoBN().to(device)
wbn  = NetWithBN().to(device)
acc_nobn = quick_train(nobn)
acc_wbn  = quick_train(wbn)

plt.figure(figsize=(7,4))
plt.plot(acc_nobn, 'r-o', ms=5, label='Without BatchNorm')
plt.plot(acc_wbn,  'b-o', ms=5, label='With BatchNorm')
plt.xlabel("Epoch"); plt.ylabel("Test Accuracy (%)")
plt.title("BatchNorm: Faster & Better Convergence")
plt.legend(); plt.grid(alpha=0.3); plt.show()


## 5.4 — Learning Rate Scheduling

In [ ]:
# ── Compare learning rate schedules ──────────────────────
schedulers_demo = {
    "StepLR (decay every 10 ep)":       [],
    "CosineAnnealing":                   [],
    "ReduceLROnPlateau":                 [],
    "OneCycleLR":                        [],
}

base_lr = 0.1
ep = list(range(50))

# StepLR
lr = base_lr
lrs_step = []
for i in ep:
    lrs_step.append(lr)
    if (i+1) % 10 == 0: lr *= 0.1

# CosineAnnealing
lrs_cosine = [base_lr * (1 + np.cos(np.pi * i / 50)) / 2 for i in ep]

# OneCycleLR (triangular with warmup)
def one_cycle(i, max_lr=0.1, epochs=50):
    if i < epochs//2:
        return max_lr * i / (epochs//2)
    return max_lr * (1 - (i - epochs//2) / (epochs//2)) + 1e-4
lrs_onecycle = [one_cycle(i) for i in ep]

plt.figure(figsize=(10, 4))
plt.plot(ep, lrs_step,     'b-o', ms=4, label='StepLR')
plt.plot(ep, lrs_cosine,   'r-',  lw=2, label='CosineAnnealing')
plt.plot(ep, lrs_onecycle, 'g-',  lw=2, label='OneCycleLR')
plt.axhline(base_lr, color='gray', ls='--', alpha=0.5, label='Constant LR')
plt.xlabel("Epoch"); plt.ylabel("Learning Rate")
plt.title("Learning Rate Schedule Comparison")
plt.legend(); plt.grid(alpha=0.3); plt.yscale('log'); plt.show()

print("Recommendation: CosineAnnealingLR is a safe default for most tasks.")
print("OneCycleLR often achieves best accuracy with proper tuning.")


## 5.5 — Early Stopping

Early stopping monitors validation loss and stops training when it stops improving — preventing overfitting and saving compute.

In [ ]:
# ── Early stopping implementation ─────────────────────────
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001, verbose=True):
        self.patience  = patience
        self.delta     = delta
        self.verbose   = verbose
        self.best_loss = float('inf')
        self.counter   = 0
        self.stop      = False
        self.best_weights = None
    
    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            if self.verbose:
                print(f"  ✅ Val loss improved to {val_loss:.4f}")
        else:
            self.counter += 1
            if self.verbose:
                print(f"  ⏳ No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.stop = True
                if self.verbose:
                    print("  🛑 Early stopping triggered!")
    
    def restore_best(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)
            print(f"  ↩️  Restored best model (val loss: {self.best_loss:.4f})")

# Demo
print("Early Stopping Demo:")
es = EarlyStopping(patience=3, verbose=True)
simulated_val_losses = [0.8, 0.7, 0.65, 0.64, 0.66, 0.67, 0.68, 0.69]
for i, loss in enumerate(simulated_val_losses):
    print(f"Epoch {i+1}: val_loss={loss:.2f}")
    es.step(loss, nn.Linear(1,1))
    if es.stop:
        break


## 5.6 — Production Training Checklist

Use this before launching any training run:

```
□ Data
  □ Normalised inputs (mean≈0, std≈1)
  □ Train/Val/Test split (e.g. 70/15/15)
  □ No data leakage from val/test into train
  □ Appropriate data augmentation

□ Model
  □ He initialisation (default in PyTorch for ReLU nets)
  □ BatchNorm in hidden layers
  □ Dropout in classifier head

□ Training
  □ Start with Adam + lr=1e-3
  □ Use a LR scheduler (CosineAnnealing or ReduceLROnPlateau)
  □ Gradient clipping (clip_grad_norm_, max=1.0) for RNNs
  □ Early stopping with patience=5-10

□ Monitoring
  □ Log train + val loss every epoch
  □ Monitor val accuracy, not just loss
  □ Save best checkpoint (not last!)

□ Debug
  □ Overfit on a small batch first (sanity check)
  □ Check that loss starts at log(n_classes) for classification
```


## 📝 Exercises

1. **Overfit first**: Take a CNN from Module 3 and overfit it on 100 samples. Once you confirm it can overfit, add regularisation to fix it. This is the most important debugging workflow.
2. **LR finder**: Implement a simple LR range test — train for one epoch, increasing LR exponentially from 1e-7 to 10. Plot loss vs LR. The optimal LR is just before the loss explodes.
3. **Checkpoint saving**: Modify `EarlyStopping` to save the model to disk using `torch.save(model.state_dict(), 'best.pth')` and reload it.
4. **TensorBoard**: Add TensorBoard logging to a training run: `from torch.utils.tensorboard import SummaryWriter`.

---

## ✅ Module 5 Summary

| Technique | When to use |
|---|---|
| Dropout | When val loss >> train loss (overfitting) |
| Weight decay (L2) | Almost always — even small values help |
| BatchNorm | After every Linear/Conv layer (before activation) |
| CosineAnnealing LR | Safe default scheduler for most tasks |
| Early stopping | Always — prevents wasted compute and overfitting |

**Next up → Module 6: Introduction to Transformers**
